# Aim 3 종단 타당도 검증 — 가·나·다·라절 분석 (v2: 정확한 컬럼명 + Aim2 동일구조 모형)

## v2에서 수정된 것
1. **KeyError 수정**: 실제 컬럼 접미사는 `__resolved`(밑줄 2개), `Q7전화`=`ADL0107A`, `Q15외출`=`ADL0115A` (서브항목 A만 존재, 부모코드 단독 없음)
2. **B2 모형을 다항 로지스틱 근사 → Aim2와 동일한 5-임계값 부분비례오즈 구조로 교체**: `b2_7item_final_mapping_with_significance.csv`에 있는 P(Y≥1)~P(Y≥5) 5개 임계값 전체 계수를 그대로 참고해, 동일한 방식(릿지 로지스틱, L2, C=1.0, 임계값별 개별 적합)으로 fold마다 재적합. 전체 데이터로 재적합했을 때 이 계수와 근사하게 나오는지 검증 셀도 포함.

## 확정 사항 (재확인)

| 항목 | 결정 |
|---|---|
| B2 예측단계 | Aim2 확정 τ규칙 (τ4=0.08, τ5=0.56, 조건부중앙값 폴백) + 5-임계값 부분비례오즈 구조 |
| 3범주 카파 안정 밴드 | Δ=0만 안정 |
| 1순위 지표(Sensitivity)에서 B2가 낮게 나와도 | 별도 규칙 도입 안 함, 있는 그대로 보고 |
| Spearman | 동점보정 (scipy 기본값 사용) |
| 부트스트랩 | B=2000 |
| 다중비교 보정 | 안 함 (가절 우선순위 체계로 대체) |
| 다절 SE 근사 | R(`lme4`+`lmerTest`) |
| 이번 범위 | 가·나·다·라만 (마절 시뮬레이션 제외) |


## 0. 환경설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
np.random.seed(42)


In [ ]:
# ===== CONFIG =====
DATA_DIR = '/content/drive/MyDrive/2026 urp/preprocessed'

TAU4, TAU5 = 0.08, 0.56          # Aim2 확정 임계값
BOOTSTRAP_B = 2000                 # 확정: B=2000
RANDOM_STATE = 42
VISIT_MAP = {2.0: 'V2', 5.0: 'V5', 7.0: 'V7'}

# ✅ 확정 (b2_7item_final_mapping_with_significance.csv, adl_wide.csv 실제 컬럼명 확인 완료)
# 순서 중요: b2_7item_final_mapping_with_significance.csv의 행 순서와 동일 (x1..x7)
ITEM_COL_MAP = {
    'Q3_toilet':         'ADL0103',            # gate 없음(raw 그대로 사용)
    'Q4_bath':            'ADL0104',            # gate 없음(raw 그대로 사용)
    'Q5_grooming':         'ADL0105',            # gate 없음(raw 그대로 사용)
    'Q6a_choose_cloth':     'ADL0106A__resolved', # gate 있음 → resolved(미실시=0) 사용
    'Q15_outing':            'ADL0115A__resolved', # gate 있음 → resolved 사용
    'Q16a_shopping':          'ADL0116A__resolved', # gate 있음 → resolved 사용
    'Q7_telephone':            'ADL0107A__resolved', # gate 있음 → resolved 사용
}
ITEM_COLS = list(ITEM_COL_MAP.values())
INDICATORS = ['A0', 'A1', 'B2']

# Aim2 확정 최종모형(전체표본 1회 적합) 계수 — 검증용 참고값
# b2_7item_final_mapping_with_significance.csv 그대로 옮김
FINAL_COEFS_REFERENCE = {
    1: {'intercept': 18.5542, 'ADL0103': -0.6125, 'ADL0104': -1.4067, 'ADL0105': -1.4270,
        'ADL0106A': -0.2913, 'ADL0115A': -0.1145, 'ADL0116A': -0.6906, 'ADL0107A': -0.4251},
    2: {'intercept': 15.3714, 'ADL0103': 0.2007, 'ADL0104': -1.6221, 'ADL0105': -1.1302,
        'ADL0106A': -0.2279, 'ADL0115A': -0.3235, 'ADL0116A': -0.6649, 'ADL0107A': -0.4611},
    3: {'intercept': 10.7059, 'ADL0103': -0.9470, 'ADL0104': -0.5963, 'ADL0105': -0.5409,
        'ADL0106A': -0.1555, 'ADL0115A': -1.0800, 'ADL0116A': -0.2962, 'ADL0107A': -0.2162},
    4: {'intercept': 5.6184, 'ADL0103': -0.9030, 'ADL0104': -0.8011, 'ADL0105': -0.7259,
        'ADL0106A': -0.2859, 'ADL0115A': -0.2077, 'ADL0116A': -0.1191, 'ADL0107A': -0.0317},
    5: {'intercept': 2.0909, 'ADL0103': -1.1319, 'ADL0104': -0.4253, 'ADL0105': -0.3292,
        'ADL0106A': 0.1839, 'ADL0115A': -0.3333, 'ADL0116A': -0.0019, 'ADL0107A': 0.0185},
}


## 1. 데이터 로드 및 Aim3 종단 분석표 구성

In [ ]:
baseline = pd.read_csv(f'{DATA_DIR}/baseline_sample.csv')
ds_wide  = pd.read_csv(f'{DATA_DIR}/ds_wide.csv')
adl_wide = pd.read_csv(f'{DATA_DIR}/adl_wide.csv')
dm       = pd.read_csv(f'{DATA_DIR}/dm_filtered.csv')

print('baseline_sample:', baseline.shape)
print('ds_wide:', ds_wide.shape)
print('adl_wide:', adl_wide.shape)
print('dm_filtered:', dm.shape)


In [ ]:
# ITEM_COL_MAP 검증
missing = [c for c in ITEM_COLS if c not in adl_wide.columns]
if missing:
    print('⚠️ 다음 컬럼을 adl_wide.csv에서 찾지 못했습니다:')
    print(missing)
    print('\n__resolved로 끝나는 전체 후보 컬럼:')
    print(sorted([c for c in adl_wide.columns if c.endswith('__resolved')]))
else:
    print('✅ 7개 항목 컬럼 전부 확인됨:', ITEM_COLS)


In [ ]:
def pivot_scores(df, value_cols, id_cols=('STUDYID', 'USUBJID')):
    """참가자×방문 long → 참가자 wide (컬럼명_V2/_V5/_V7)로 변환"""
    d = df[df['VISITNUM'].isin([2.0, 5.0, 7.0])].copy()
    d['visit'] = d['VISITNUM'].map(VISIT_MAP)
    keep = list(id_cols) + ['visit'] + value_cols
    d = d[keep].drop_duplicates(subset=list(id_cols) + ['visit'])
    wide = d.pivot(index=list(id_cols), columns='visit', values=value_cols)
    wide.columns = [f'{c}_{v}' for c, v in wide.columns]
    return wide.reset_index()

# 7개 항목 wide
items_wide = pivot_scores(adl_wide, ITEM_COLS)

# A0 / A1 wide
a0a1_wide = pivot_scores(adl_wide, ['A0_harmonized', 'A1_2015_stage'])
a0a1_wide = a0a1_wide.rename(columns={
    'A0_harmonized_V2': 'A0_pred_V2', 'A0_harmonized_V5': 'A0_pred_V5', 'A0_harmonized_V7': 'A0_pred_V7',
    'A1_2015_stage_V2': 'A1_pred_V2', 'A1_2015_stage_V5': 'A1_pred_V5', 'A1_2015_stage_V7': 'A1_pred_V7',
})

# 실측 DS wide
ds_pivot = pivot_scores(ds_wide, ['ds_stage', 'ds_total'])

aim3 = items_wide.merge(a0a1_wide, on=['STUDYID', 'USUBJID'], how='left')
aim3 = aim3.merge(ds_pivot, on=['STUDYID', 'USUBJID'], how='inner')

if 'ARM' in dm.columns:
    aim3 = aim3.merge(dm[['USUBJID', 'ARM']], on='USUBJID', how='left')

print('merge 후:', aim3.shape)


In [ ]:
# Aim3 종단 표본 필터
req_v2_items = [f'{c}_V2' for c in ITEM_COLS]
before = len(aim3)
aim3 = aim3.dropna(subset=req_v2_items)
aim3 = aim3[aim3['ds_stage_V2'].notna() & (aim3['ds_stage_V5'].notna() | aim3['ds_stage_V7'].notna())]
print(f'필터 전 {before}명 → 필터 후 {len(aim3)}명')
print(aim3['STUDYID'].value_counts())


## 2. B2 모형 — Aim2와 동일한 5-임계값 부분비례오즈 구조

**원 모형 구조**: $P(Y\ge1), P(Y\ge2), P(Y\ge3), P(Y\ge4), P(Y\ge5)$ 각각을 **별도의 릿지 로지스틱(L2, C=1.0)**으로
적합(부분비례오즈 — 임계값마다 기울기가 다를 수 있음). 개별 단계확률은 뺄셈 조립:

$P(Y=0)=1-P(Y\ge1),\quad P(Y=k)=P(Y\ge k)-P(Y\ge k+1)\ (k=1..4),\quad P(Y=5)=P(Y\ge5)$

**정보누출 방지**: 매 fold마다 held-out 시험을 제외한 나머지 2개 시험의 V2 자료로만 5개 모형을 적합하고
계수를 고정, held-out 시험의 V2·V5·V7에 그대로 적용.

In [ ]:
CUTPOINTS = [1, 2, 3, 4, 5]

def fit_b2_model(train_df):
    """Aim2와 동일 구조: 임계값별(1~5) 개별 릿지 로지스틱(L2, C=1.0)"""
    X = train_df[[f'{c}_V2' for c in ITEM_COLS]].values
    y = train_df['ds_stage_V2'].astype(int).values

    models = {}
    for k in CUTPOINTS:
        target = (y >= k).astype(int)
        if target.sum() == 0 or target.sum() == len(target):
            models[k] = None  # 해당 fold에 해당 임계값 이상/이하 사례가 아예 없는 경우
            continue
        m = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=2000)
        m.fit(X, target)
        models[k] = m
    return models


def predict_b2(models, X):
    """5개 임계값 모형 → 개별 단계확률(뺄셈 조립, 클리핑) → τ 판정규칙 적용"""
    n = X.shape[0]
    p_ge = np.zeros((n, 6))  # p_ge[:,k] = P(Y>=k), k=0..5 (k=0은 항상 1)
    p_ge[:, 0] = 1.0
    for k in CUTPOINTS:
        m = models[k]
        if m is None:
            # 해당 fold 훈련자료에 해당 임계값 이상/이하 사례가 없었던 경우: 인접 임계값으로 대체
            p_ge[:, k] = p_ge[:, k - 1]
        else:
            p_ge[:, k] = m.predict_proba(X)[:, 1]

    # 뺄셈 조립 (단조성 깨질 수 있어 클리핑 후 재정규화)
    full_proba = np.zeros((n, 6))
    full_proba[:, 5] = p_ge[:, 5]
    for k in [4, 3, 2, 1]:
        full_proba[:, k] = p_ge[:, k] - p_ge[:, k + 1]
    full_proba[:, 0] = 1 - p_ge[:, 1]
    full_proba = np.clip(full_proba, 0, None)
    row_sums = full_proba.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    full_proba = full_proba / row_sums

    p_y5 = full_proba[:, 5]
    p_y_ge4 = full_proba[:, 4] + full_proba[:, 5]
    cum = np.cumsum(full_proba, axis=1)

    pred_stage = np.zeros(n, dtype=int)
    for i in range(n):
        if p_y5[i] > TAU5:
            pred_stage[i] = 5
        elif p_y_ge4[i] > TAU4:
            pred_stage[i] = 4
        else:
            pred_stage[i] = int(np.argmax(cum[i] >= 0.5))  # 조건부중앙값

    expected_stage = full_proba @ np.arange(6)  # 라절: 기대단계
    return pred_stage, expected_stage, full_proba


### 2-1. 검증 셀 — 전체표본 1회 적합 시 확정 계수와 근사한지 확인

Aim3 표본(V2, 3개 시험 전체)으로 fold 없이 1회 적합해서, `b2_7item_final_mapping_with_significance.csv`의
확정 계수와 부호·크기가 대략 일치하는지 확인합니다. **완전히 같은 숫자는 안 나올 수 있음** — Aim2 원 모형은
"in_common_comparison_sample"(n=2,196) 전체를 썼지만, 여기 Aim3 표본은 wk12/wk24 관측까지 요구해서
표본이 더 작기 때문. 부호와 대략적 크기, 특히 유의했던 항목(Q3·Q15 등)의 방향이 같은지가 핵심 체크포인트.

In [ ]:
full_models = fit_b2_model(aim3.dropna(subset=['ds_stage_V2']))

compare_rows = []
for k in CUTPOINTS:
    m = full_models[k]
    if m is None:
        continue
    fitted_coef = dict(zip(ITEM_COLS, m.coef_[0]))
    fitted_intercept = m.intercept_[0]
    ref = FINAL_COEFS_REFERENCE[k]
    for item_key, col in ITEM_COL_MAP.items():
        base_code = col.replace('__resolved', '')
        compare_rows.append({
            'cutpoint': f'P(Y>={k})',
            'item': item_key,
            'fitted_coef(Aim3표본)': round(fitted_coef[col], 4),
            'reference_coef(Aim2확정)': ref.get(base_code, np.nan),
        })

compare_df = pd.DataFrame(compare_rows)
display(compare_df)


### 2-2. 진단 — 왜 P(Y≥1)/P(Y≥2)만 크게 어긋나는가

가설: (1) 이 표본은 wk12/wk24까지 관측된 완주자만 남긴 표본이라 Aim2의 원래 baseline 전체 표본보다
좁고 경증 쪽으로 약간 치우쳐 있음 → Y=0(무증상) 사례가 특히 희소해져 P(Y≥1) 추정이 불안정.
(2) Q3·Q4·Q5(화장실·목욕·몸단장)는 서로 상관이 높아 릿지 계수가 표본에 민감(다중공선성).
아래에서 표본크기, 임계값별 클래스 비율, 항목간 상관을 직접 확인.

In [ ]:
fit_sample = aim3.dropna(subset=['ds_stage_V2'])
print(f'적합에 쓰인 표본 크기: n={len(fit_sample)} (Aim2 확정계수는 n=2,196 기준)')
print()

print('임계값별 Y>=k 클래스 비율 (해당 값이 1에 가까울수록 Y=0/1이 희소하다는 뜻):')
y = fit_sample['ds_stage_V2'].astype(int)
for k in [1, 2, 3, 4, 5]:
    ratio = (y >= k).mean()
    n_below = (y < k).sum()
    print(f'  P(Y>={k}): {ratio:.3f}  (Y<{k}인 사례 n={n_below})')

print()
print('7개 항목(V2) 상관행렬 — Q3/Q4/Q5가 서로 유독 높으면 다중공선성 가설 지지:')
item_v2_cols = [f'{c}_V2' for c in ITEM_COLS]
corr = fit_sample[item_v2_cols].corr()
corr.index = list(ITEM_COL_MAP.keys())
corr.columns = list(ITEM_COL_MAP.keys())
display(corr.round(2))


In [ ]:
for held_out in aim3['STUDYID'].unique():
    print(held_out, '시험 held-out 시 훈련표본 크기:',
          (aim3['STUDYID'] != held_out).sum())


In [ ]:
for col in ['B2_pred_V2', 'B2_pred_V5', 'B2_pred_V7',
            'B2_expected_V2', 'B2_expected_V5', 'B2_expected_V7']:
    aim3[col] = np.nan

trials = aim3['STUDYID'].unique().tolist()

for held_out in trials:
    train_df = aim3[aim3['STUDYID'] != held_out].dropna(subset=['ds_stage_V2'])
    models = fit_b2_model(train_df)

    test_mask = aim3['STUDYID'] == held_out
    for visit in ['V2', 'V5', 'V7']:
        feat_cols = [f'{c}_{visit}' for c in ITEM_COLS]
        sub = aim3.loc[test_mask, feat_cols]
        valid = sub.notna().all(axis=1)
        if valid.sum() == 0:
            continue
        X = sub.loc[valid].values
        pred, expected, _ = predict_b2(models, X)
        idx = sub.loc[valid].index
        aim3.loc[idx, f'B2_pred_{visit}'] = pred
        aim3.loc[idx, f'B2_expected_{visit}'] = expected

    print(f'fold(held-out={held_out}): train n={len(train_df)}, test n={test_mask.sum()}')

aim3[['STUDYID', 'B2_pred_V2', 'B2_pred_V5', 'B2_pred_V7']].head()


## 3. 가절 — 일차 종단평가 (단계변화 비교)

$\Delta_{24} = V7-V2$ (1차), $\Delta_{12} = V5-V2$ (2차)를 분리 계산합니다.

In [ ]:
def compute_deltas(df, span='24wk'):
    v_from, v_to = ('V2', 'V7') if span == '24wk' else ('V2', 'V5')
    out = pd.DataFrame(index=df.index)
    out['STUDYID'] = df['STUDYID']
    out['delta_DS'] = df[f'ds_stage_{v_to}'] - df[f'ds_stage_{v_from}']
    out['delta_TDS'] = df[f'ds_total_{v_to}'] - df[f'ds_total_{v_from}']
    for m in INDICATORS:
        out[f'delta_{m}'] = df[f'{m}_pred_{v_to}'] - df[f'{m}_pred_{v_from}']
    return out

delta24 = compute_deltas(aim3, '24wk').dropna(subset=['delta_DS'])
delta12 = compute_deltas(aim3, '12wk').dropna(subset=['delta_DS'])
print(f'24주 비교 가능 n={len(delta24)}, 12주 비교 가능 n={len(delta12)}')


### 3-1. 단계변화량 일치도 (MAE, Spearman) — 3순위 지표

In [ ]:
def eval_change_agreement(delta_df):
    rows = []
    for m in INDICATORS:
        sub = delta_df.dropna(subset=[f'delta_{m}'])
        mae = (sub['delta_DS'] - sub[f'delta_{m}']).abs().mean()
        rho, p = spearmanr(sub['delta_DS'], sub[f'delta_{m}'])  # 동점보정 기본 적용
        rows.append({'indicator': m, 'n': len(sub), 'MAE': mae,
                      'spearman_rho': rho, 'spearman_p': p})
    return pd.DataFrame(rows)

print('[24주]')
display(eval_change_agreement(delta24))
print('[12주]')
display(eval_change_agreement(delta12))


### 3-2. 3범주(악화/안정/호전) 가중카파 — 2순위 지표 (안정=Δ0만, 확정)

In [ ]:
def categorize_3cat(delta):
    """확정: Δ=0만 안정. -1이하=호전(0), 0=안정(1), +1이상=악화(2)"""
    return np.where(delta <= -1, 0, np.where(delta == 0, 1, 2))

def weighted_kappa_3cat(delta_df):
    rows = []
    for m in INDICATORS:
        sub = delta_df.dropna(subset=[f'delta_{m}'])
        yt = categorize_3cat(sub['delta_DS'].values)
        yp = categorize_3cat(sub[f'delta_{m}'].values)
        kappa = cohen_kappa_score(yt, yp, weights='linear')
        rows.append({'indicator': m, 'n': len(sub), 'weighted_kappa': kappa})
    return pd.DataFrame(rows)

print('[24주]')
display(weighted_kappa_3cat(delta24))
print('[12주]')
display(weighted_kappa_3cat(delta12))


In [ ]:
from sklearn.metrics import confusion_matrix

CAT_LABELS = ['호전(Δ≤-1)', '안정(Δ=0)', '악화(Δ≥+1)']

def confusion_matrices_3cat(delta_df):
    """지표별 3범주 혼동행렬: 행=실측(delta_DS), 열=예측(delta_{m})"""
    mats = {}
    for m in INDICATORS:
        sub = delta_df.dropna(subset=[f'delta_{m}'])
        yt = categorize_3cat(sub['delta_DS'].values)
        yp = categorize_3cat(sub[f'delta_{m}'].values)
        cm = confusion_matrix(yt, yp, labels=[0, 1, 2])
        cm_df = pd.DataFrame(
            cm,
            index=[f'실측_{l}' for l in CAT_LABELS],
            columns=[f'예측_{l}' for l in CAT_LABELS]
        )
        mats[m] = cm_df
    return mats

def print_confusion_matrices(delta_df, span_label):
    print(f'===== {span_label} 혼동행렬 (행=실측 DS, 열=예측) =====')
    for m, cm_df in confusion_matrices_3cat(delta_df).items():
        n = cm_df.values.sum()
        print(f'\n[{m}]  n={n}')
        display(cm_df)
        # 행(row) 기준 정규화 = 실측 범주별 예측 분포(재현율 비슷한 관점)
        display(cm_df.div(cm_df.sum(axis=1), axis=0).round(3))

print_confusion_matrices(delta24, '24주')
print_confusion_matrices(delta12, '12주')

In [ ]:
!pip install koreanize-matplotlib
import koreanize_matplotlib

In [ ]:
import matplotlib.pyplot as plt

def plot_confusion_heatmaps(delta_df, span_label):
    mats = confusion_matrices_3cat(delta_df)
    fig, axes = plt.subplots(1, len(INDICATORS), figsize=(5*len(INDICATORS), 4))
    for ax, m in zip(axes, INDICATORS):
        cm_df = mats[m]
        im = ax.imshow(cm_df.values, cmap='Blues')
        ax.set_xticks(range(3)); ax.set_xticklabels(CAT_LABELS, rotation=30, ha='right')
        ax.set_yticks(range(3)); ax.set_yticklabels(CAT_LABELS)
        ax.set_xlabel('예측'); ax.set_ylabel('실측')
        ax.set_title(f'{m} ({span_label})')
        for i in range(3):
            for j in range(3):
                ax.text(j, i, cm_df.values[i, j], ha='center', va='center')
    plt.tight_layout()
    plt.show()

plot_confusion_heatmaps(delta24, '24주')
plot_confusion_heatmaps(delta12, '12주')

In [ ]:
"""### 3-2-보완. EDA — 왜 3범주 가중카파가 낮은가 (델타 분포 + 카파역설 진단)"""

# (1) 델타 원값 분포 그 자체 — 몇 명이 -3,-2,-1,0,+1,... 인지
def delta_distribution_summary(delta_df, span_label):
    cols = ['delta_DS'] + [f'delta_{m}' for m in INDICATORS]
    print(f'===== {span_label}: Δ값 원 분포(정수) =====')
    for col in cols:
        vc = delta_df[col].dropna().value_counts().sort_index()
        print(f'\n[{col}]  n={vc.sum()}')
        print(vc)
    display(delta_df[cols].describe().T)

delta_distribution_summary(delta24, '24주')
delta_distribution_summary(delta12, '12주')

# (2) 원시일치율(raw agreement) vs 가중카파 vs 3범주 주변분포(%) — 카파역설 직접 확인
def kappa_diagnostics_3cat(delta_df, span_label):
    mats = confusion_matrices_3cat(delta_df)
    kappa_df = weighted_kappa_3cat(delta_df).set_index('indicator')
    rows = []
    for m in INDICATORS:
        cm = mats[m].values
        n = cm.sum()
        po = np.trace(cm) / n                      # 원시일치율
        row_marg = cm.sum(axis=1) / n               # 실측(DS) 주변분포
        col_marg = cm.sum(axis=0) / n               # 예측(지표) 주변분포
        pe = (row_marg * col_marg).sum()             # 우연일치확률(unweighted)
        kappa_unw = (po - pe) / (1 - pe)             # 비교용 unweighted kappa
        rows.append({
            'indicator': m, 'n': n,
            '원시일치율(%)': po*100,
            '우연일치확률pe(%)': pe*100,
            'kappa(unweighted)': kappa_unw,
            'weighted_kappa': kappa_df.loc[m, 'weighted_kappa'],
            '실측_호전%': row_marg[0]*100, '실측_안정%': row_marg[1]*100, '실측_악화%': row_marg[2]*100,
            '예측_호전%': col_marg[0]*100, '예측_안정%': col_marg[1]*100, '예측_악화%': col_marg[2]*100,
        })
    out = pd.DataFrame(rows).round(2)
    print(f'\n===== {span_label}: 원시일치율 vs pe vs 카파 + 3범주 주변분포 =====')
    display(out)
    return out

kd24 = kappa_diagnostics_3cat(delta24, '24주')
kd12 = kappa_diagnostics_3cat(delta12, '12주')

### 3-3. 악화 이진판정 — 1순위 지표 (사전확정 준거: TDS≥2)

준거(실측)는 총점 기준, 검사(지표)는 단계변화≥1 기준 — 비대칭임에 유의.

In [ ]:
def binary_worsening_eval(delta_df, criterion_col='delta_TDS', criterion_thr=2):
    W_all = (delta_df[criterion_col] >= criterion_thr).astype(int)
    rows = []
    eval_targets = INDICATORS + ['DS_stage(실측)']   # ← 추가된 부분

    for m in eval_targets:
        pred_col = 'delta_DS' if m == 'DS_stage(실측)' else f'delta_{m}'   # ← 추가된 부분
        idx = delta_df[pred_col].notna()
        W = W_all[idx].values
        What = (delta_df.loc[idx, pred_col] >= 1).astype(int).values

        TP = int(((What == 1) & (W == 1)).sum()); FN = int(((What == 0) & (W == 1)).sum())
        TN = int(((What == 0) & (W == 0)).sum()); FP = int(((What == 1) & (W == 0)).sum())

        sens = TP / (TP + FN) if (TP + FN) > 0 else np.nan
        spec = TN / (TN + FP) if (TN + FP) > 0 else np.nan
        ppv  = TP / (TP + FP) if (TP + FP) > 0 else np.nan
        fpr  = 1 - spec if not np.isnan(spec) else np.nan

        sens_ci = proportion_confint(TP, TP + FN, method='wilson') if (TP + FN) > 0 else (np.nan, np.nan)
        spec_ci = proportion_confint(TN, TN + FP, method='wilson') if (TN + FP) > 0 else (np.nan, np.nan)

        rows.append({'indicator': m, 'n': int(idx.sum()), 'TP': TP, 'FN': FN, 'TN': TN, 'FP': FP,
                      'sensitivity': sens, 'sens_95CI': sens_ci,
                      'specificity': spec, 'spec_95CI': spec_ci,
                      'PPV': ppv, 'FPR': fpr})
    return pd.DataFrame(rows)
print('[24주, 1차 준거: TDS 변화≥2]')
binary_24wk_primary = binary_worsening_eval(delta24, 'delta_TDS', 2)
display(binary_24wk_primary)


In [ ]:
# 민감도분석: 원문 표의 4개 준거 정의 전부 병기
SENS_CRITERIA = {
    'TDS>=2 (1차)': ('delta_TDS', 2),
    'TDS>=1':       ('delta_TDS', 1),
    'stage>=1':     ('delta_DS', 1),
    'stage>=2':     ('delta_DS', 2),
}

for label, (col, thr) in SENS_CRITERIA.items():
    print(f'--- {label} (24주) ---')
    display(binary_worsening_eval(delta24, col, thr))


### 3-4. 지표간 비교 — 부트스트랩 CI(MAE 차이) + McNemar(이진판정 차이)

In [ ]:
def bootstrap_mae_diff(delta_df, m1='B2', m2='A0', B=BOOTSTRAP_B, seed=RANDOM_STATE):
    sub = delta_df.dropna(subset=[f'delta_{m1}', f'delta_{m2}'])
    err1 = (sub['delta_DS'] - sub[f'delta_{m1}']).abs().values
    err2 = (sub['delta_DS'] - sub[f'delta_{m2}']).abs().values
    diff = err1 - err2
    rng = np.random.default_rng(seed)
    n = len(diff)
    boots = np.array([diff[rng.integers(0, n, n)].mean() for _ in range(B)])
    ci_low, ci_high = np.percentile(boots, [2.5, 97.5])
    return diff.mean(), ci_low, ci_high

for m2 in ['A0', 'A1']:
    mean_diff, ci_l, ci_h = bootstrap_mae_diff(delta24, 'B2', m2)
    sig = '유의' if (ci_l > 0 or ci_h < 0) else '유의하지 않음'
    print(f'MAE(B2) - MAE({m2}) [24주] = {mean_diff:.3f}, 95% CI [{ci_l:.3f}, {ci_h:.3f}] → {sig}')


In [ ]:
def mcnemar_compare(delta_df, m1, m2, criterion_col='delta_TDS', criterion_thr=2):
    idx = delta_df[f'delta_{m1}'].notna() & delta_df[f'delta_{m2}'].notna()
    W  = (delta_df.loc[idx, criterion_col] >= criterion_thr).astype(int)
    W1 = (delta_df.loc[idx, f'delta_{m1}'] >= 1).astype(int)
    W2 = (delta_df.loc[idx, f'delta_{m2}'] >= 1).astype(int)
    correct1 = (W1 == W).astype(int)
    correct2 = (W2 == W).astype(int)
    table = pd.crosstab(correct1, correct2)
    result = mcnemar(table.values, exact=False, correction=True)
    return table, result

for m2 in ['A0', 'A1']:
    table, result = mcnemar_compare(delta24, 'B2', m2)
    print(f'--- B2 vs {m2} (24주, TDS≥2 준거) ---')
    print(table)
    print(f'McNemar chi2={result.statistic:.3f}, p={result.pvalue:.4f}\n')


### 3-5. Bland-Altman (단계변화, 단위 동일한 것만)

In [ ]:
!pip install koreanize-matplotlib
import koreanize_matplotlib

In [ ]:
def bland_altman(delta_df, m='B2', span_label='24주'):
    sub = delta_df.dropna(subset=[f'delta_{m}'])
    d = sub[f'delta_{m}'] - sub['delta_DS']
    mean_diff = d.mean()
    loa = 1.96 * d.std()
    avg = (sub[f'delta_{m}'] + sub['delta_DS']) / 2

    plt.figure(figsize=(6, 4))
    plt.scatter(avg, d, alpha=0.3, s=10)
    plt.axhline(mean_diff, color='red', linestyle='--', label=f'평균편차={mean_diff:.2f}')
    plt.axhline(mean_diff + loa, color='gray', linestyle=':')
    plt.axhline(mean_diff - loa, color='gray', linestyle=':')
    plt.xlabel('평균 (예측·실측 단계변화)')
    plt.ylabel('차이 (예측-실측)')
    plt.title(f'Bland-Altman: {m} vs 실측 DS 단계변화 ({span_label})')
    plt.legend()
    plt.tight_layout()
    plt.show()
    return mean_diff, mean_diff - loa, mean_diff + loa

for m in INDICATORS:
    bland_altman(delta24, m, '24주')


## 4. 나절 — 연속 출력 반응성 비교

실측 DS 총점 vs B2 기대단계(연속). SRM만 단독 판단하지 않고 변화상관도 병행 보고합니다.

In [ ]:
def compute_change(df, kind, visit='V7'):
    if kind == 'DS_total':
        base_col, change_col = 'ds_total_V2', f'ds_total_{visit}'
    elif kind == 'B2_expected':
        base_col, change_col = 'B2_expected_V2', f'B2_expected_{visit}'
    else:
        raise ValueError(kind)
    base = df[base_col]
    change = df[change_col] - base
    return base, change.dropna()

def src_srm(df, kind, visit='V7'):
    base, change = compute_change(df, kind, visit)
    src = change.mean() / base.std()
    srm = change.mean() / change.std()
    return src, srm, change

src_ds, srm_ds, chg_ds = src_srm(aim3, 'DS_total', 'V7')
src_b2, srm_b2, chg_b2 = src_srm(aim3, 'B2_expected', 'V7')
print(f'실측 DS총점 [24주]: raw mean change={chg_ds.mean():.3f}, SRC={src_ds:.3f}, SRM={srm_ds:.3f}')
print(f'B2 기대단계 [24주]: raw mean change={chg_b2.mean():.3f}, SRC={src_b2:.3f}, SRM={srm_b2:.3f}')

common_idx = chg_ds.index.intersection(chg_b2.index)
change_corr, change_corr_p = spearmanr(chg_ds.loc[common_idx], chg_b2.loc[common_idx])
print(f'\nΔ실측DS총점 vs ΔB2기대단계 상관: rho={change_corr:.3f} (p={change_corr_p:.4f}, n={len(common_idx)})')


In [ ]:
def bootstrap_srm_diff(df, kind1='B2_expected', kind2='DS_total', visit='V7',
                        B=BOOTSTRAP_B, seed=RANDOM_STATE):
    _, _, chg1 = src_srm(df, kind1, visit)
    _, _, chg2 = src_srm(df, kind2, visit)
    common = chg1.index.intersection(chg2.index)
    c1, c2 = chg1.loc[common].values, chg2.loc[common].values
    rng = np.random.default_rng(seed)
    n = len(c1)
    boots = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        boots[b] = (c1[idx].mean() / c1[idx].std()) - (c2[idx].mean() / c2[idx].std())
    return np.percentile(boots, [2.5, 97.5])

ci = bootstrap_srm_diff(aim3)
print(f'SRM(B2기대단계) - SRM(실측DS총점) 95% 부트스트랩 CI [24주]: [{ci[0]:.3f}, {ci[1]:.3f}]')
print('※ 이 CI만으로 "B2가 더 반응적"이라 단정하지 않음 — 위 변화상관과 함께 해석')


In [ ]:
"""### 4-보완. EDA — SRM·SRC 분산 분해: 기저치분산 vs 변화량분산, 어느 쪽이 압축됐나"""

from scipy.stats import levene

def variance_decomposition_table(df, visit='V7', span_label='24주'):
    base_ds, chg_ds = compute_change(df, 'DS_total', visit)
    base_b2, chg_b2 = compute_change(df, 'B2_expected', visit)

    rows = []
    for name, base, chg in [('실측 DS총점', base_ds, chg_ds), ('B2 기대단계', base_b2, chg_b2)]:
        rows.append({
            'indicator': name,
            '기저치_mean': base.mean(), '기저치_SD': base.std(),
            '변화량_mean': chg.mean(), '변화량_SD': chg.std(),
            'SRC(=변화량mean/기저치SD)': chg.mean()/base.std(),
            'SRM(=변화량mean/변화량SD)': chg.mean()/chg.std(),
        })
    out = pd.DataFrame(rows).round(4)
    print(f'===== {span_label} 분산 분해 =====')
    display(out)

    base_ratio = base_b2.std() / base_ds.std()
    chg_ratio = chg_b2.std() / chg_ds.std()
    mean_ratio = chg_b2.mean() / chg_ds.mean()

    print()
    print('기저치_SD 비(B2/DS)  =', round(base_ratio, 3))
    print('변화량_SD 비(B2/DS)  =', round(chg_ratio, 3), '  <- 이게 SRM 분모, 훨씬 작으면 압축 가설 지지')
    print('변화량_mean 비(B2/DS) =', round(mean_ratio, 3), '  <- 신호(분자) 자체 차이')

    common = chg_ds.index.intersection(chg_b2.index)
    stat, p = levene(chg_ds.loc[common], chg_b2.loc[common])
    print()
    print('Levene 등분산검정 (변화량 SD 차이 유의성): stat=', round(stat, 3), 'p=', round(p, 4))

    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].boxplot([base_ds, base_b2], labels=['DS총점', 'B2기대단계'])
    ax[0].set_title(f'기저치 분포 ({span_label})')
    ax[1].boxplot([chg_ds.loc[common], chg_b2.loc[common]], labels=['DS총점', 'B2기대단계'])
    ax[1].set_title(f'변화량 분포 ({span_label})')
    plt.tight_layout()
    plt.show()

    return out

variance_decomposition_table(aim3, 'V7', '24주')
variance_decomposition_table(aim3, 'V5', '12주')

## 5. 다절 — 반복측정 혼합모형 (R: lme4 + lmerTest)

$Y_{it} = \beta_0 + \beta_1 \cdot \text{Visit}_t + \beta_2 \cdot \text{Trial}_i + \beta_3 \cdot \text{Treatment}_i + b_i + \varepsilon_{it}$

기저 SD로 표준화한 실측 DS총점, B2 기대단계 각각에 대해 별도 적합.

> **실행 환경 안내**: 모형 적합(주분석 + 5-1 공분산구조 민감도분석)은 Colab의 rpy2 대신 **로컬 R**에서 수행합니다.
> (Colab에서 rpy2로 `lme4`를 설치하면 컴파일 의존성 문제로 실패하는 경우가 많아, 안정적인 로컬 R 실행으로 전환했습니다.)
> 이 노트북(Python/Colab)에서는 **long-format 데이터를 만들어 CSV로 내보내는 것까지만** 담당하고,
> 실제 `lmer()` 적합은 별도 스크립트 `run_aim3_covariance_sensitivity.R`에서 실행합니다.

In [ ]:
def build_long_for_mixed(df, kind):
    """kind: 'DS_total' 또는 'B2_expected'"""
    col_prefix = 'ds_total' if kind == 'DS_total' else 'B2_expected'
    baseline_sd = df[f'{col_prefix}_V2'].std()

    rows = []
    for visit in ['V2', 'V5', 'V7']:
        tmp = df[['USUBJID', 'STUDYID', f'{col_prefix}_{visit}']].copy()
        tmp = tmp.rename(columns={f'{col_prefix}_{visit}': 'Y_raw'})
        tmp['Visit'] = visit
        if 'ARM' in df.columns:
            tmp['Treatment'] = df['ARM']
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)
    long_df['Y_std'] = long_df['Y_raw'] / baseline_sd
    long_df = long_df.dropna(subset=['Y_std'])
    return long_df

long_ds_total = build_long_for_mixed(aim3, 'DS_total')
long_b2_exp   = build_long_for_mixed(aim3, 'B2_expected')
print(long_ds_total.shape, long_b2_exp.shape)
long_ds_total.head()


### 5-0. 장기 포맷(long-format) 데이터 → CSV 내보내기

아래 셀에서 만든 `long_ds_total` / `long_b2_exp`를 CSV로 저장한 뒤,
로컬 `preprocessed` 폴더로 다운로드하세요. (원본 preprocessed CSV에는 B2 예측값이 없어 이 내보내기가 꼭 필요합니다.)

CSV 저장 후에는 로컬에서 **`run_aim3_covariance_sensitivity.R`**을 실행하면 됩니다.
이 스크립트가 주분석(random intercept)과 5-1 공분산구조 민감도분석(random slope + AIC 비교)을 모두 수행하고,
결과를 `aim3_다절_결과_R_local.txt`로 저장합니다.

In [ ]:
long_ds_total.to_csv(f'{DATA_DIR}/aim3_long_ds_total.csv', index=False)
long_b2_exp.to_csv(f'{DATA_DIR}/aim3_long_b2_expected.csv', index=False)

print('저장 완료:')
print(f'  {DATA_DIR}/aim3_long_ds_total.csv')
print(f'  {DATA_DIR}/aim3_long_b2_expected.csv')
print()
print('다음 단계: 위 두 CSV를 로컬 preprocessed 폴더로 다운로드한 뒤,')
print('run_aim3_covariance_sensitivity.R 스크립트를 로컬 R에서 실행하세요.')

### 5-1. 공분산 구조 민감도분석 (random slope / AIC 비교)

`run_aim3_covariance_sensitivity.R`에서 함께 실행됩니다 (범주형 Visit random slope는 2시점-전용 참가자 때문에
식별 불가하여, 연속시간 `time_num`(0/12/24)으로 대체 — 스크립트 내 주석 참고).

결과 요약(발표/슬라이드용 참고, `aim3_다절_결과_R_local.txt`에서 발췌):

| | AIC (random intercept) | AIC (+ random slope) |
|---|---|---|
| 실측 DS총점 | 13938.86 | 13845.13 |
| B2 기대단계 | 12781.85 | 12579.88 |

→ 두 지표 모두 random slope 모형의 AIC가 뚜렷이 낮아 compound symmetry(참가자 간 동일 기울기 가정)를 기각.

## 6. 라절 — 예측 기대단계 해석 (용도 구분 확인)

정수예측단계(`B2_pred_*`)는 가절(일치도·악화판정)에, 기대단계(`B2_expected_*`)는 나절(연속 반응성)에,
실측 총점(`ds_total_*`)은 준거로만 — 이미 위 코드에서 각 절마다 맞는 컬럼을 쓰고 있는지 아래에서 확인.

In [ ]:
print('가절에서 쓴 컬럼: delta_B2 (= B2_pred_V7 - B2_pred_V2, 정수, τ규칙)')
print('나절에서 쓴 컬럼: B2_expected_V2/V5/V7 (연속, 확률가중평균)')
print()
print('샘플 비교 (같은 사람의 정수예측단계 vs 기대단계는 다를 수 있음 — 정상):')
aim3[['USUBJID', 'B2_pred_V2', 'B2_expected_V2', 'B2_pred_V7', 'B2_expected_V7']].dropna().head(10)
